In [1]:
import joblib
import pandas as pd
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv("../data/cloudburst_dataset.csv")
df

,Cloud_Top_Height,Cloud_Base_Height,Optical_Thickness,Rainfall,Humidity,Temperature,Pressure,cloud_burst,source_type
0,14102.922471,4478.254022,81.458788,113.860000,98.456707,2.460000,831.976032,1,actual_based
1,16280.349921,3282.266452,100.754718,145.754353,96.215862,5.351109,806.439829,1,actual_based
2,14610.191174,3540.533728,79.712916,113.860000,100.342319,2.460000,832.275850,1,actual_based
3,14442.462510,5226.559823,68.037289,136.020961,98.155915,5.775977,847.044026,1,actual_based
4,15792.328643,2259.156281,65.791433,113.860000,100.163245,4.246368,847.358308,1,actual_based
...,...,...,...,...,...,...,...,...,...
9995,12416.550137,4418.751713,39.545973,158.989305,64.571139,33.088374,987.295359,0,normal_synthetic
9996,15214.536284,4427.930940,30.847472,138.896527,63.000000,13.006152,985.756576,1,normal_synthetic
9997,12942.719214,1080.787694,58.312846,137.858924,63.000000,30.256021,954.880262,1,normal_synthetic
9998,14245.382958,3919.624936,45.259220,116.310489,67.982034,23.534713,992.333455,0,normal_synthetic


In [3]:
X = df.drop(columns=["cloud_burst", "source_type"])
y = df["cloud_burst"]
X

,Cloud_Top_Height,Cloud_Base_Height,Optical_Thickness,Rainfall,Humidity,Temperature,Pressure
0,14102.922471,4478.254022,81.458788,113.860000,98.456707,2.460000,831.976032
1,16280.349921,3282.266452,100.754718,145.754353,96.215862,5.351109,806.439829
2,14610.191174,3540.533728,79.712916,113.860000,100.342319,2.460000,832.275850
3,14442.462510,5226.559823,68.037289,136.020961,98.155915,5.775977,847.044026
4,15792.328643,2259.156281,65.791433,113.860000,100.163245,4.246368,847.358308
...,...,...,...,...,...,...,...
9995,12416.550137,4418.751713,39.545973,158.989305,64.571139,33.088374,987.295359
9996,15214.536284,4427.930940,30.847472,138.896527,63.000000,13.006152,985.756576
9997,12942.719214,1080.787694,58.312846,137.858924,63.000000,30.256021,954.880262
9998,14245.382958,3919.624936,45.259220,116.310489,67.982034,23.534713,992.333455


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="gpu_hist",
    random_state=42
)

In [7]:
param_grid = {
    "n_estimators": [10, 50, 100, 200, 300, 400, 500, 1000],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.001, 0.01, 0.05, 0.1, 0.2, 0.5],
    "subsample": [0.8, 1.0]
}

In [8]:
grid_search = GridSearchCV(
    estimator=xgb_clf,
    param_grid=param_grid,
    scoring="accuracy",
    cv=3,
    verbose=1,
    n_jobs=-1
)

In [9]:
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Fitting 3 folds for each of 288 candidates, totalling 864 fits
Best Parameters: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
Best CV Score: 0.830500275987632


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [19:47:06] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)


In [10]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
cr = classification_report(y_test, y_pred)
print(cr)

Test Accuracy: 0.8355
              precision    recall  f1-score   support

           0       0.80      0.86      0.83       932
           1       0.87      0.81      0.84      1068

    accuracy                           0.84      2000
   macro avg       0.84      0.84      0.84      2000
weighted avg       0.84      0.84      0.84      2000



/usr/local/lib/python3.12/dist-packages/xgboost/core.py:2676: UserWarning: [19:47:06] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  if len(data.shape) != 1 and self.num_features() != data.shape[1]:
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:729: UserWarning: [19:47:06] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [ ]:
joblib.dump(grid_search.best_estimator_, "../models/xgboost_model.pkl")

['xgboost_model.pkl']

In [ ]:
# Test Loaded Model
loaded_model = joblib.load("../models/xgboost_model.pkl")

test_val = [[4000,16.16,82.50,1,78.4,26.5,935.81]] # Not CB
# test_val = [[16000,21.35,3.33,583,78.1,25.8,935.86]] # CB

test_val = scaler.transform(test_val)
print("Test Value Prediction:", loaded_model.predict(test_val))
print("Test Value Prediction Probability:", loaded_model.predict_proba(test_val))

Test Value Prediction: [0]
Test Value Prediction Probability: [[0.6875775 0.3124225]]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:2676: UserWarning: [19:47:44] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  if len(data.shape) != 1 and self.num_features() != data.shape[1]:
